In [1]:
%reload_ext autoreload
%autoreload 2

In [1]:
# Import Dependencies
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import jax
import jax.numpy as jp

# Set environment variables for EGL rendering BEFORE importing mujoco
os.environ['MUJOCO_GL'] = 'egl'
os.environ["CUDA_VISIBLE_DEVICES"] = "1" 

import mujoco
from mujoco import mjx

# Add the project to Python path
sys.path.append('/home/duckoid/Downloads/mujoco_playground')


import mediapy as media
import mujoco_playground
from mujoco_playground import registry

In [4]:
env = registry.load("RobcoArm")
env_cfg = registry.get_default_config("RobcoArm")
print("RobcoArm environment loaded successfully!")
print(f"Observation space size: {env.observation_size}")
print(f"Action space size: {env.action_size}")

state = env.reset(0)
env.get_end_effector_position(state.data)

RobcoArm environment loaded successfully!
Observation space size: 18
Action space size: 6


Array([-0.10019994,  0.        ,  1.4090999 ], dtype=float32)

In [4]:
env.mjx_model.actuator_ctrlrange

Array([[-4.7124,  4.7124],
       [-4.7124,  4.7124],
       [-4.7124,  4.7124],
       [-4.7124,  4.7124],
       [-4.7124,  4.7124],
       [-4.7124,  4.7124]], dtype=float32)

In [6]:
env = registry.load("PandaRobotiqPushCube")
env.mjx_model.actuator_ctrlrange

Array([[-1.  ,  1.  ],
       [-1.  ,  1.  ],
       [-1.  ,  1.  ],
       [-1.  ,  1.  ],
       [-1.  ,  1.  ],
       [-1.  ,  1.  ],
       [-1.  ,  1.  ],
       [ 0.  ,  0.82]], dtype=float32)

In [ ]:
jit_reset = jax.jit(env.reset)
jit_step = jax.jit(env.step)

In [ ]:
state = jit_reset(jax.random.PRNGKey(0))
rollout = [state]

end_effector_target_rew = []
ground_collision_rew = []
self_collision_rew = []
energy_rew = []

f = 0.5
for i in range(1000):
  action = []
  for j in range(env.action_size):
    action.append(
        jp.sin(
            state.data.time * 2 * jp.pi * f + j * 2 * jp.pi / env.action_size
        )
    )
  action = jp.array(action)
  state = jit_step(state, action)
  end_effector_target_rew.append(env._config.reward_config.scales["end_effector_target"] * state.metrics['end_effector_target'])
  ground_collision_rew.append(env._config.reward_config.scales["ground_collision"] * state.metrics['ground_collision'])
  self_collision_rew.append(env._config.reward_config.scales["self_collision"] * state.metrics['self_collision'])
  energy_rew.append(env._config.reward_config.scales["energy"] * state.metrics['energy'])
  rollout.append(state)

frames = env.render(rollout)
media.show_video(frames, fps=1.0 / env.dt)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(4, 1, figsize=(8, 10), sharex=True)

axes[0].plot(end_effector_target_rew)
axes[0].set_title("End Effector Target Reward")

axes[1].plot(ground_collision_rew)
axes[1].set_title("Ground Collision Reward")

axes[2].plot(self_collision_rew)
axes[2].set_title("Self Collision Reward")

axes[3].plot(energy_rew)
axes[3].set_title("Energy Reward")
axes[3].set_xlabel("Timestep")

for ax in axes:
    ax.set_ylabel("Reward")

plt.tight_layout()
plt.show()
